In [1]:
import os
import torch
import torch.optim as optim

# Impor dari semua modul yang relevan
from config import (
    LEARNING_RATE,
    N_USER_POINTS,
    SHP_PATH,
    USER_JSON_PATH,
    USE_SHP_FOR_USER,
)
from device_config import info_device, DEVICE
from model import PolicyNetwork
from data_loader import (
    load_evac_candidates_shp,
    generate_user_coords_from_shp,
    load_user_coords,
)
from data_utils import load_flood_polygons

# Impor file baru yang sudah disederhanakan
from flood_trainer import train_rl_model, evaluate_model

In [2]:
info_device()

# --- Pastikan semua path file sudah benar ---
DATA_DIR = "./kota-surabaya"
SAVE_DIR = "./hasil_training_banjir"
os.makedirs(SAVE_DIR, exist_ok=True)

FLOOD_SHP_PATH = os.path.join(DATA_DIR, "Genangan Revisi Lagi.shp")
EVAC_SHP_PATH = os.path.join(DATA_DIR, "Titik_Evakuasi_2.shp")
USER_AREA_SHP_PATH = SHP_PATH

# --- Memuat semua data ke dalam variabel ---
print("Memuat data dari Shapefile...")

# 1. Muat poligon banjir terlebih dahulu untuk mendapatkan CRS referensi.
flood_polygons_gdf = load_flood_polygons(FLOOD_SHP_PATH)
if flood_polygons_gdf is None:
    raise ValueError("🔴 Gagal memuat data poligon banjir. Proses dihentikan.")
target_crs = flood_polygons_gdf.crs  # Dapatkan CRS dari sini

# 2. Muat titik evakuasi dan berikan CRS target untuk disamakan.
evac_candidates = load_evac_candidates_shp(EVAC_SHP_PATH, target_crs=target_crs)

# 3. Muat data pengguna.
if USE_SHP_FOR_USER:
    user_shp_full_path = USER_AREA_SHP_PATH
    if not os.path.dirname(user_shp_full_path):
        user_shp_full_path = os.path.join(
            DATA_DIR, os.path.basename(user_shp_full_path)
        )
    user_coords = generate_user_coords_from_shp(user_shp_full_path, n=N_USER_POINTS)
else:
    user_coords = load_user_coords(USER_JSON_PATH)

# Validasi data setelah pemuatan
if not evac_candidates or not user_coords:
    raise ValueError(
        "🔴 Gagal memuat data kandidat evakuasi atau pengguna. Proses dihentikan."
    )
else:
    print("\n✅ Semua data berhasil dimuat dengan CRS yang selaras.")

✅ Perangkat aktif: CPU
Memuat data dari Shapefile...
✅ Shapefile banjir berhasil dimuat dari: ./kota-surabaya\Genangan Revisi Lagi.shp (6 poligon)
⚠️ Peringatan: CRS tidak ditemukan di Titik_Evakuasi_2.shp. Mengasumsikan WGS84 (EPSG:4326).
✅ 225 kandidat evakuasi berhasil dimuat dari: Titik_Evakuasi_2.shp
✅ 50 koordinat pengguna dimuat dari JSON.

✅ Semua data berhasil dimuat dengan CRS yang selaras.


In [ ]:
policy_model = PolicyNetwork(input_dim=6, output_dim=1).to(DEVICE)
optimizer = optim.Adam(policy_model.parameters(), lr=LEARNING_RATE)

# Jumlah episode training
NUM_EPISODES = 1

# Panggil fungsi training
trained_model = train_rl_model(
    model=policy_model,
    optimizer=optimizer,
    num_episodes=NUM_EPISODES,
    user_coords=user_coords,
    evac_candidates=evac_candidates,
    flood_gdf=flood_polygons_gdf,
)


🚀 Memulai training untuk 1 episode...


In [ ]:
if trained_model:
    evaluate_model(
        model=trained_model,
        user_coords=user_coords,
        evac_candidates=evac_candidates,
        flood_gdf=flood_polygons_gdf,
    )

In [ ]:
if trained_model:
    model_save_path = os.path.join(SAVE_DIR, "banjir_rl_model.pt")
    torch.save(trained_model.state_dict(), model_save_path)
    print(f"\n💾 Model terlatih disimpan di: {model_save_path}")

print("\n🎉 Proses Selesai.")